# Imports

In [ ]:
import kagglehub
import os
from PIL import Image, ImageDraw, ImageFont
import string
import torch
import torch.nn as nn
from tqdm import tqdm


from CRNN import CRNN
from dataset import SROIEDataset
from utils import parse_line, encode

device = "cuda" if torch.cuda.is_available() else "cpu"

# Download Dataset

In [ ]:
ds_path = kagglehub.dataset_download("urbikn/sroie-datasetv2")

print("Path to dataset files:", ds_path)

## Inspect an image

In [ ]:
i = 2
train_path_img = os.path.join(ds_path, "SROIE2019/train/img")
test_path = os.path.join(ds_path, "SROIE2019/test")
img_name = os.listdir(train_path_img)[i]
file_path = os.path.join(train_path_img, img_name)
img = Image.open(file_path)
draw = ImageDraw.Draw(img)

# Define font
try:
    font = ImageFont.truetype("arial.ttf", 14)
except:
    font = ImageFont.load_default()

train_path_entities = os.path.join(ds_path, "SROIE2019/train/entities")
with open(os.path.join(train_path_entities, 
                   os.listdir(train_path_entities)[i])) as f:
    text = f.readlines()

train_path_boxes =  os.path.join(ds_path, "SROIE2019/train/box")

with open(os.path.join(train_path_boxes, 
                   os.listdir(train_path_boxes)[i])) as f:
    lines = f.readlines()

for line in lines:
    coords, text = parse_line(line)

    # Convert to list of tuples
    polygon = [(coords[i], coords[i+1]) for i in range(0, 8, 2)]

    # Draw box
    draw.polygon(polygon, outline="red")

    # Draw label (top-left corner)
    draw.text((coords[0], coords[1] - 12), text, fill="blue", font=font)
display(img)

# Character Encoding

In [ ]:
BLANK = 0 # Reserved for loss function

CHARS = "#" + " " + string.ascii_letters + string.digits + ".,:-/"
char_to_idx = {c: i+1 for i, c in enumerate(CHARS)}  # shift by +1
idx_to_char = {i: c for c, i in char_to_idx.items()}

# Define model, dataset and hyperparameters

In [ ]:
NUM_EPOCHS = 1
learning_rate = 1e-3
batch_size = 8

model = CRNN(len(CHARS) + 1).to(device)
criterion = nn.CTCLoss(blank=BLANK, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

dataset = SROIEDataset(train_path_img, train_path_boxes, train_path_entities)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)


# Training Loop

In [ ]:
for epoch in tqdm(range(NUM_EPOCHS)):
    for imgs, texts in loader:
        imgs = imgs.to(device)

        encoded = [encode(t, char_to_idx) for t in texts]

        # compute T from a dummy forward to know the sequence length
        with torch.no_grad():
            T = model(imgs[:1]).size(1)

        # filter out samples whose target is longer than T (CTC requires T >= L)
        valid = [(img, enc) for img, enc in zip(imgs, encoded)
                 if enc is not None and len(enc) <= T]
        if not valid:
            continue

        imgs_valid, encoded_valid = zip(*valid)
        imgs_valid = torch.stack(imgs_valid).to(device)

        target_lengths = torch.tensor([len(e) for e in encoded_valid])
        targets = torch.cat(encoded_valid).to(device)

        preds = model(imgs_valid)       # (B, T, C)
        preds = preds.log_softmax(2)
        preds = preds.permute(1, 0, 2)  # (T, B, C) for CTC

        input_lengths = torch.full(
            size=(imgs_valid.size(0),),
            fill_value=preds.size(0),
            dtype=torch.long,
        )

        loss = criterion(preds, targets, input_lengths, target_lengths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}: {loss.item():.4f}")